In [ ]:
import re  # For preprocessing
import pandas as pd  # For data handling
from time import time  # To time our operations
from collections import defaultdict  # For word frequency
from gensim.models import Word2Vec
from gensim.test.utils import common_texts, get_tmpfile
from gensim.models import Phrases
import nltk
from nltk import bigrams
import gensim

from gensim.models.phrases import Phrases, Phraser

import multiprocessing

from gensim.models import Word2Vec

import spacy  # For preprocessing

from gensim.models import KeyedVectors

import logging  # Setting up the loggings to monitor gensim
logging.basicConfig(format="%(levelname)s - %(asctime)s: %(message)s", datefmt= '%H:%M:%S', level=logging.INFO)

In [ ]:
#---------Read all concept wordnet and mesh
dataConceptMeSH = pd.read_csv('../Data_extracted/data_complet_MeSH(V3).csv', error_bad_lines= False, encoding='utf-8', names = ["c_ui", "d_name", "d_ui", "rel", "scopeNote", "tree", "word_list"])
dataConceptWN = pd.read_csv('../Data_extracted/data_complet_WN(v3).csv', error_bad_lines= False, encoding='utf-8', names = ["c_name", "synset", "hyponymes", "hypernyms", "definition", "exemple", "list_word"])

In [ ]:
dataConceptMeSH.head()

In [ ]:
dataConceptWN.head()

In [ ]:
#---------Read all concept a verifie wn and mesh
dataConceptMeSH_a_verifie = pd.read_csv('../Data_extracted/data_MeSH_concept_verifie(V1).csv', error_bad_lines= False, encoding='utf-8', names = ["c_ui", "d_name", "d_ui", "rel", "scopeNote", "tree", "word_list"])
dataConceptWN_a_verifie = pd.read_csv('../Data_extracted/data_wn_concept_verifie(V1).csv', error_bad_lines= False, encoding='utf-8', names = ["c_name", "synset", "hyponymes", "hypernyms", "definition", "exemple", "list_word"])

In [ ]:
print('all concept MeSH : ', dataConceptMeSH.shape[0])
print('all concept WN : ', dataConceptWN.shape[0])

In [ ]:
#-----------check missing item in csv
dataConceptMeSH.isnull().sum()
dataConceptWN.isnull().sum()

In [ ]:
#----------full all empty item
dataConceptMeSH = dataConceptMeSH.dropna().reset_index(drop=True)
dataConceptMeSH.isnull().sum()

dataConceptWN = dataConceptWN.dropna().reset_index(drop=True)
dataConceptWN.isnull().sum()

In [ ]:
nlp = spacy.load('en', disable=['ner', 'parser']) # disabling Named Entity Recognition for speed

def cleaning(doc):
    # Lemmatizes and removes stopwords
    # doc needs to be a spacy Doc object
    txt = [token.lemma_ for token in doc if not token.is_stop]
    # Word2Vec uses context words to learn the vector representation of a target word,
    # if a sentence is only one or two words long,
    # the benefit for the training is very small
    if len(txt) > 2:
        return ' '.join(txt)

In [ ]:
#----------get all definitions of MeSH
def get_all_definition_MeSH(dataConceptMeSH):
    definitions_MeSH = []
    len_MeSH = dataConceptMeSH.shape[0]
    for i in range(len_MeSH):
        definitions_MeSH.append(dataConceptMeSH.iloc[i].scopeNote)
    definitions_MeSH = definitions_MeSH[1:] #eliminate frist cause it is the head of csv: 'scopeNote'
    return definitions_MeSH

#----------get all definitions of WN
def get_all_definition_WN(dataConceptWN):
    definitions_WN = []
    len_WN = dataConceptWN.shape[0]
    for i in range(len_WN):
        definitions_WN.append(dataConceptWN.iloc[i].definition)
    definitions_WN = definitions_WN[1:] #eliminate frist cause it is the head of csv: 'definition'
    return definitions_WN

In [ ]:
definitions_MeSH = get_all_definition_MeSH(dataConceptMeSH)
definitions_WN = get_all_definition_WN(dataConceptWN)
print(definitions_MeSH[:10])
print(definitions_WN[:10])

In [ ]:

#----------brief cleaning definition concept of MeSH and WN
def brief_cleaning(definition):
    return (re.sub("[^A-Za-z']+", ' ', str(row)).lower() for row in definition)

brief_cleaning_MeSH = brief_cleaning(definitions_MeSH)
brief_cleaning_WN = brief_cleaning(definitions_WN)


In [ ]:
#-----------cleaning up and build pipeline for trainning after(MeSH)
def cleaning_up_and_build_pip(brief_cleaning):
    t = time()
    txt = [cleaning(doc) for doc in nlp.pipe(brief_cleaning, batch_size=5000, n_threads=-1)]
    print('Time to clean up definition : {} mins'.format(round((time() - t) / 60, 2)))
    return txt

txt_MeSH = cleaning_up_and_build_pip(brief_cleaning_MeSH)
txt_WN = cleaning_up_and_build_pip(brief_cleaning_WN)

In [ ]:
#------------remove missing concept and duplicates concept in MeSH
def remove_missing_duplicates(txt):
    df_clean = pd.DataFrame({'clean': txt})
    df_clean = df_clean.dropna().drop_duplicates()
    df_clean.shape
    return df_clean

df_clean_MeSH = remove_missing_duplicates(txt_MeSH)
df_clean_WN = remove_missing_duplicates(txt_WN)


In [ ]:
#----------separate MeSH definition cleaning to word
def to_word(df_clean):
    return [row.split() for row in df_clean['clean']]

sent_MeSH = to_word(df_clean_MeSH)
sent_WN = to_word(df_clean_WN)


In [ ]:
#------------control out of vocabulary.这个还没结束
sent_MeSH.append(['UNK'])
print(sent_MeSH[-1])

sent_WN.append(['UNK'])
print(sent_WN[-1])


Before feeding the corpus to word2vec for training, need to combine some of the common word group into a single word, which will be very helpful for the accuracy of the training.

For exemple : ['New', 'York'] become ['New York']
and if second time to run this code can trun :
['New York', 'Times'] become ['New York Times']

公式放不进来

In [ ]:
def compose_word_to_word_group(list_sent_word):
    return Phrases(list_sent_word, min_count=30, progress_per=10000)

phrases_MeSH = compose_word_to_word_group(sent_MeSH)
phrases_MeSH = compose_word_to_word_group(sent_WN)


Make all word in a sentence to bigram in a sentence.
But this not use in suit of code.

In [ ]:
def to_bigram(list_sent_word):
    sentences = []
    for word in list_sent_word:
        sentences.append(list(bigrams(word)))
    return sentences

sentences_MeSH = to_bigram(sent_MeSH)
sentences_WN = to_bigram(sent_WN)


Count all frequence of word in MeSH and WN

In [ ]:
def count_word_freq(list_sent_word):
    word_freq = defaultdict(int)
    for sent in list_sent_word:
        for i in sent:
            word_freq[i] += 1
    return word_freq

word_freq_MeSH = count_word_freq(sent_MeSH)
word_freq_WN = count_word_freq(sent_WN)

print(len(word_freq_MeSH))
print(len(word_freq_WN))

In [ ]:
#---------Test
def ten_words_most_frequent(word_freq):
    return sorted(word_freq, key=word_freq.get, reverse=True)[:10]
    
print('ten words most frequent in MeSH : ', ten_words_most_frequent(word_freq_MeSH))
print()
print('ten words most frequent in WN : ', ten_words_most_frequent(word_freq_WN))



In [ ]:
# Count the number of cores in a computer
# for workers in next step
cores = multiprocessing.cpu_count() 

In [ ]:
#------------Create model empty(skip-gram + hierarchical softmax)
def create_model(min_count, window, size, sample, alpha, min_alpha, negative, workers, hs, sg):
    return Word2Vec(min_count=min_count, window=window, size=size, sample=sample, 
                     alpha=alpha,min_alpha=min_alpha, negative=negative,workers=workers, 
                     hs = hs, sg = sg)

w2v_model_WN = create_model(min_count=20,
                     window=2,
                     size=300,
                     sample=6e-5, 
                     alpha=0.03, 
                     min_alpha=0.0007, 
                     negative=20,
                     workers=cores-1, 
                     hs = 1,
                     sg = 1)

w2v_model_MeSH = create_model(min_count=20,
                     window=2,
                     size=300,
                     sample=6e-5, 
                     alpha=0.03, 
                     min_alpha=0.0007, 
                     negative=20,
                     workers=cores-1, 
                     hs = 1,
                     sg = 1)

In [ ]:
#---------if runtime error, update parametre of w2v_model
#---------in this step, we use all sentences to build the vocab
#---------in case of OOV(out of vocabulary)

def build_vocab(model_empty, list_sent_word):
    t = time()
    model_empty.build_vocab(list_sent_word, progress_per=10000)
    print('Time to build vocab : {} mins'.format(round((time() - t) / 60, 2)))

build_vocab(w2v_model_MeSH, sent_MeSH)
build_vocab(w2v_model_WN, sent_WN)


In [ ]:
#------------lenght of vocabulary MeSH and WN
print(len(w2v_model_MeSH.wv.vocab))
print(len(w2v_model_WN.wv.vocab))

In [ ]:
#----------show word in vocabulary MeSH
#for word in w2v_model_MeSH.wv.vocab:
    #print(word)

#----------show word in vocabulary WN
#for word in w2v_model_WN.wv.vocab:
    #print(word)
    
#methode to get word exact
#w2v_model.wv.vocab['word']

In [ ]:
#-----------if need, 70% of sentence to train. and 30% for test.
def divise_trainData_and_testdata(list_sent_word):
    train_sents = []
    test_sents = []

    for i, sent in enumerate(list_sent_word):
        if i <= int(len(list_sent_word)*0.7):
            train_sents.append(sent)
        else:
            test_sents.append(sent)
    return train_sents, test_sents

train_sents_MeSH, test_sents_MeSH = divise_trainData_and_testdata(sent_MeSH)
train_sents_WN, test_sents_WN = divise_trainData_and_testdata(sent_WN)

print(len(sent_MeSH), len(train_sents_MeSH), len(test_sents_WN))
print(len(sent_WN), len(train_sents_WN), len(test_sents_WN))


In [ ]:
#-------------train model with all data
def train_model(model, train_data):
    t_MeSH = time()
    model.train(train_data, total_examples = model.corpus_count, epochs = 30, report_delay = 1)
    print('Time to train the model MeSH: {} mins'.format(round((time() - t_MeSH) / 60, 2)))

train_model(w2v_model_MeSH, sent_MeSH)
train_model(w2v_model_WN, sent_WN)


In [ ]:
#------------save model with format model
def save_model_format_model(model, name_file_suffix_model):
    model.save(name_file_suffix_model)
    model.wv.save_word2vec_format(name_file_suffix_model, binary=True)

save_model_format_model(w2v_model_MeSH, "word2vec_MeSH_all_sents.model")
save_model_format_model(w2v_model_WN, "word2vec_WN_all_sents.model")


In [ ]:
#----------save model with format kv
def save_model_format_kv(model, name_file_suffix_kv):
    model.wv.save(name_file_suffix_kv)
    
save_model_format_kv(w2v_model_MeSH, 'wordvector_MeSH.kv')
save_model_format_kv(w2v_model_WN, 'wordvector_WN.kv')


In [ ]:
def load_model_format_kv(name_file_suffix_kv, memory_map):
    return KeyedVectors.load("wordvector_MeSH.kv", mmap = memory_map)

wv_MeSH = load_model_format_kv("wordvector_MeSH.kv", 'r')
wv_WN = load_model_format_kv("wordvector_WN.kv", 'r')


In [ ]:
#--------------Test
vector_MeSH = wv_MeSH['neoplasm']
vector_WN = wv_WN['neoplasm']
#print(vector_MeSH)
#print(vector_WN)

In [ ]:
#--------load model and test
model_MeSH = gensim.models.KeyedVectors.load_word2vec_format("word2vec_MeSH_all_sents.model", binary=True, unicode_errors='ignore')
model_WN = gensim.models.KeyedVectors.load_word2vec_format("word2vec_WN_all_sents.model", binary=True, unicode_errors='ignore')

关于这个结果 我们可以看到neoplasm在mesh内最近的词是tumor 但是在wn内最近的词是
virus 要好好想想这是为甚

In [ ]:
for item in model_MeSH.most_similar('neoplasm'):
    print(item[0], item[1])
    
#print(model_MeSH.most_similar('neoplasm'))

print()

for item in model_WN.most_similar('neoplasm'):
    print(item[0], item[1])
    
#print(model_WN.most_similar('neoplasm'))

In [ ]:
print(model_MeSH.similarity('neoplasm', 'neoplasm'))

print(model_WN.similarity('neoplasm', 'neoplasm'))

In [ ]:
#------------Accuracy of model gensim

In [ ]:
#-----------Test

#----------get 1000 definition of MeSH to sentences 1 than 1000 to sentences 2
#sent1 = definitions_MeSH[1:2]
#sent2 = definitions_MeSH[100:101]
#----------comprared definition of same concept : 'Neoplasm' in MeSH and WN
#sent1 = ['New abnormal growth of tissue. Malignant neoplasms show a greater degree of anaplasia and have the properties of invasion and metastasis, compared to benign neoplasms. ']
#sent2 = ['an abnormal new mass of tissue that serves no purpose']
#print(sent1[0])
#print(sent2[0])

In [ ]:
#-------------update new word not in vocabulary of model

#count the frenquenties of evry word in sentences1 and sentences2
#frequenties_sent1 = {}
#frequenties_sent2 = {}

#for sen in sent1:
#    fdist = nltk.FreqDist(nltk.word_tokenize(sen))
#    for word, frequency in fdist.most_common(50):
#        frequenties_sent1[word] = frequency
#        #print(u'{}:{}'.format(word, frequency))
        
#for sen in sent2:
#    fdist = nltk.FreqDist(nltk.word_tokenize(sen))
#    for word, frequency in fdist.most_common(50):
#        frequenties_sent2[word] = frequency
#print(sent1)
#print(sent2)
#print(frequenties)

In [ ]:
#---------merge two dictionary, not use
def merge_two_dicts(x, y):
    """Given two dicts, merge them into a new dict as a shallow copy."""
    z = x.copy()
    z.update(y)
    return z

In [ ]:
sent_MeSH = definitions_MeSH
sent_WN = definitions_WN
print(sent_MeSH[0])
print(sent_WN[0])

In [ ]:
import nltk

STOP = set(nltk.corpus.stopwords.words("english"))

class Sentence:
    
    def __init__(self, sentence):
        self.raw = sentence
        normalized_sentence = sentence.replace("‘", "'").replace("’", "'")
        self.tokens = [t.lower() for t in nltk.word_tokenize(normalized_sentence)]
        self.tokens_without_stop = [t for t in self.tokens if t not in STOP]

In [ ]:
#---------set the sentence to a instance Sentence: raw sentence, normalized sentence, tokens, tokens without stopwords
sentences1 = []
sentences2 = []
for i in range(len(sent_MeSH)):
    sentences1.append(Sentence(sent_MeSH[i]))
    
for i in range(len(sent_WN)):
    sentences2.append(Sentence(sent_WN[i]))

In [ ]:
print([sentences1[10].raw])
print([sentences2[10].raw])
print()
print(sentences1[10].tokens)
print(sentences2[10].tokens)
print()
print(sentences1[10].tokens_without_stop)
print(sentences2[10].tokens_without_stop)

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def remove_first_principal_component(X):
    svd = TruncatedSVD(n_components=1, n_iter=7, random_state=0)
    svd.fit(X)
    pc = svd.components_
    XX = X - X.dot(pc.transpose()) * pc
    return XX

def run_sif_benchmark(sentences1, sentences2, model1, model2, freqs1 = {}, freqs2 = {}, use_stoplist=False, a=0.001): 
    total_freq1 = sum(freqs1.values())
    total_freq2 = sum(freqs2.values())
    
    embeddings = []
    
    # SIF requires us to first collect all sentence embeddings and then perform 
    # common component analysis.
    for (sent1, sent2) in zip(sentences1, sentences2): 
        
        #----------tokens of sentence1 and sentence2
        tokens1 = sent1.tokens_without_stop if use_stoplist else sent1.tokens
        tokens2 = sent2.tokens_without_stop if use_stoplist else sent2.tokens

        tokens1 = [token for token in tokens1 if token in model1]
        tokens2 = [token for token in tokens2 if token in model2]

        #---------control of OOV 看下文章如何处理那些不在词库的单词
        if len(tokens1) == 0:
            tokens1 = ['building']
            
        if len(tokens2) == 0:
            tokens2 = ['building']
        
        #---------weights of sentences's words
        weights1 = [a/(a+freqs1.get(token,0)/total_freq1) for token in tokens1]    
        
        weights2 = [a/(a+freqs2.get(token,0)/total_freq2) for token in tokens2]
        
        #---------embedding1 sentences
        embedding1 = np.average([model1[token] for token in tokens1], axis=0, weights=weights1)
        embedding2 = np.average([model2[token] for token in tokens2], axis=0, weights=weights2)

        embeddings.append(embedding1)
        embeddings.append(embedding2)
        
    embeddings = remove_first_principal_component(np.array(embeddings))
    sims = [cosine_similarity(embeddings[idx*2].reshape(1, -1), 
                              embeddings[idx*2+1].reshape(1, -1))[0][0] 
            for idx in range(int(len(embeddings)/2))]

    return sims

In [ ]:
#----------Test 
#sim = run_sif_benchmark(sentences1, sentences2, model, freqs=word_freq, use_stoplist=True, a=0.001)

In [ ]:
def trun_negatif_number_to_positif_number(number):
    number_sim = []
    for i in range(len(number)):
        if number[i] >= 0:
            number_sim.append(number[i])
        else:
            number_sim.append(-number[i])
    return number_sim

In [ ]:
#----------get all definitions of MeSH
def get_all_definition_with_name_MeSH(dataConceptMeSH):
    definitions_MeSH = {}
    len_MeSH = dataConceptMeSH.shape[0]
    for i in range(len_MeSH):
        definitions_MeSH[dataConceptMeSH.iloc[i].d_name] = dataConceptMeSH.iloc[i].scopeNote
    #del definitions_MeSH['scopeNote'] #eliminate frist cause it is the head of csv: 'scopeNote'
    return definitions_MeSH

#----------get all definitions of WN
def get_all_definition_with_name_WN(dataConceptWN):
    definitions_WN = {}
    len_WN = dataConceptWN.shape[0]
    for i in range(len_WN):
        definitions_WN[dataConceptWN.iloc[i].c_name] = dataConceptWN.iloc[i].definition
    #del definitions_MeSH['scopeNote'] #eliminate frist cause it is the head of csv: 'scopeNote'
    return definitions_WN

In [ ]:
dic_MeSH = get_all_definition_with_name_MeSH(dataConceptMeSH)

In [ ]:
import operator as op
print(dic_MeSH['Calcimycin'])
op.eq(dic_MeSH['Calcimycin'], all_info[0][0])

In [ ]:
dic_WN = get_all_definition_with_name_WN(dataConceptWN)


In [ ]:
print(dic_WN['entity'])
value = dic_WN['entity']

In [ ]:
list(dic_WN.keys())[list(dic_WN.values()).index(value)]


In [ ]:
dic_MeSH_verifie = get_all_definition_with_name_MeSH(dataConceptMeSH_a_verifie)
dic_WN_verifie = get_all_definition_with_name_WN(dataConceptWN_a_verifie)


In [ ]:

print(dic_MeSH_verifie['Abattoirs'])
print(dic_WN_verifie['abattoir'])

a = dic_MeSH_verifie['Abattoirs']
aa = [Sentence(a)]
#print(aa)
b = dic_WN_verifie['abattoir']
bb = [Sentence(b)]
#print(bb)


eq1 = 'Abattoirs'
eq2 = 'abattoir'
def lemma_and_stemm_word(word):
    porter_stemmer = PorterStemmer()
    lemmatizer = WordNetLemmatizer()
    word_lemma = lemmatizer.lemmatize(word)
    return porter_stemmer.stem(word_lemma)

#print(word_stemm)
#print(op.eq(word_stemm, eq2))

#if((op.eq(lemma_and_stemm_word(eq1), eq2) == True) or ((eq1, eq2) == True)):
#    print("rrrrr")

sim = run_sif_benchmark(aa, bb, model_MeSH, model_WN, freqs1=word_freq_MeSH, freqs2=word_freq_WN, use_stoplist=True, a=0.001)
print(trun_negatif_number_to_positif_number(sim))

In [ ]:
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.stem.porter import PorterStemmer

In [ ]:

found_definition_WN_by_MeSH_name = []
not_found_definition_WN_by_MeSH_name = []


l_name_found_MeSH = []
l_name_found_WN = []

for key in dic_MeSH_verifie:
    try:
        original_key_found = dic_WN_verifie[key]
    except KeyError:
        try:
            stemm_key_found = dic_WN_verifie[lemma_and_stemm_word(key)]
        except KeyError:
            not_found_definition_WN_by_MeSH_name.append(key)
        else:
            l_name_found_MeSH.append(key)
            l_name_found_WN.append(lemma_and_stemm_word(key))
            found_definition_WN_by_MeSH_name.append(stemm_key_found)
    else:
        l_name_found_MeSH.append(key)
        l_name_found_WN.append(key)
        found_definition_WN_by_MeSH_name.append(original_key_found)
        
print(len(found_definition_WN_by_MeSH_name)) 
print(len(not_found_definition_WN_by_MeSH_name))
print(len(l_name_found_MeSH))
print(len(l_name_found_WN))


In [ ]:
a = dic_MeSH_verifie['Abattoirs']
aa = [Sentence(a)]
print(aa)
b = dic_WN_verifie['abattoir']
bb = [Sentence(b)]
print(bb)


In [ ]:
from nltk.corpus import wordnet as wn

In [ ]:
big_info = []
for i in range(len(l_name_found_MeSH)):
    five_info = []
    
    trun_sent_MeSH = [Sentence(dic_MeSH_verifie[l_name_found_MeSH[i]])]
    
    five_info.append(l_name_found_MeSH[i])
    five_info.append(dic_MeSH_verifie[l_name_found_MeSH[i]])
    five_info.append(trun_sent_MeSH[0].tokens_without_stop)
    
    trun_sent_WN = [Sentence(dic_WN_verifie[l_name_found_WN[i]])]
    five_info.append(l_name_found_WN[i])
    five_info.append(dic_WN_verifie[l_name_found_WN[i]])
    five_info.append(trun_sent_WN[0].tokens_without_stop)
    
    sim = run_sif_benchmark(trun_sent_MeSH, trun_sent_WN, model_MeSH, model_WN, freqs1=word_freq_MeSH, freqs2=word_freq_WN, use_stoplist=True, a=0.001)
    
    five_info.append(trun_negatif_number_to_positif_number(sim))
    
    big_info.append(five_info)


In [ ]:
#------------find concept dont have hyponyme.
l_no_hyponyme_WN = []
print(big_info[1][0])
for i in range(len(big_info)):
    a = dataConceptWN[dataConceptWN['c_name'] == big_info[i][3]]
    if(op.eq(a.hyponymes.tolist()[0], '[]')):
        l_no_hyponyme.append(a)


In [ ]:
ju = []
for i in range(len(l_no_hyponyme)):
    aaa = l_no_hyponyme[i].values.tolist()
    ju.append(aaa[0])


In [ ]:
sss = pd.DataFrame(ju, columns = ['c_name','synset','hyponymes', 'hypernyms', 'definition', 'exemple', 'list_word'])
sss.to_csv(r'no_hypo_wn.csv', index = None, header = True)

In [ ]:
#------------find concept dont have hyponyme.
l_hyponyme = []
print(big_info[1][3])
for i in range(len(big_info)):
    a = dataConceptWN[dataConceptWN['c_name'] == big_info[i][3]]
    if(op.ne(a.hyponymes.tolist()[0], '[]')):
        l_hyponyme.append(a)

In [ ]:
ju = []
for i in range(len(l_hyponyme)):
    aaa = l_hyponyme[i].values.tolist()
    ju.append(aaa[0])

In [ ]:
sss = pd.DataFrame(ju, columns = ['c_name','synset','hyponymes', 'hypernyms', 'definition', 'exemple', 'list_word'])
sss.to_csv(r'has_hypo_wn.csv', index = None, header = True)

In [ ]:
#------------ line separate.

print(big_info[1][3])

dataConceptWN.head()

l_no_hyponyme = []
for i in range(len(big_info)):
    name = big_info[i][3]
    c = dataConceptWN[dataConceptWN['c_name'] == name]
    #print(c.hyponymes.tolist())
    #print(op.eq(c.hyponymes.tolist()[0], '[]'))
    #print(len(c.hyponymes.tolist()[0]))
    #print(c.hyponymes.tolist()[0])
    print(c.hyponymes == '[]')
    #if op.eq(c.hyponymes.tolist()[0], '[]') == True:
    if (len(c.hyponymes.tolist()[0]) == 2):
        #print(len(c.hyponymes.tolist()[0]))
        #print(c.hyponymes.tolist()[0])
        #l_no_hyponyme.append(big_info[i])
        #print(c.hyponymes.tolist())
        l_no_hyponyme.append(c.values.tolist())
    else:
        continue
"""
print(len(l_no_hyponyme))

data_no_hypo = []
for i in range(len(l_no_hyponyme)):
    for j in range(len(l_no_hyponyme[i])):
        data_no_hypo.append(l_no_hyponyme[i][j])
        
aaa = pd.DataFrame(data_no_hypo, columns = ['c_name','synset','hyponymes', 'hypernyms', 'definition', 'exemple', 'list_word'])
aaa.to_csv(r'no_hypo_wn.csv', index = None, header = True)
"""

In [ ]:
#-------------Find concept dont have hyponyme.
l_wn = []
for i in range(len(big_info)):
    name = big_info[i][3]
    c = dataConceptWN[dataConceptWN['c_name'] == name]
    if op.eq(c.hyponymes.tolist()[0], '[]') == True:
        l_wn.append(c.values.tolist())

In [ ]:
data = []
for i in range(len(l_no_hyponyme)):
    for j in range(len(l_no_hyponyme[i])):
        data.append(l_no_hyponyme[i][j])
        
#print(l_wn[0][0])
#print(l_wn[0])


print(data[0])

aaa = pd.DataFrame(data, columns = ['c_name','synset','hyponymes', 'hypernyms', 'definition', 'exemple', 'list_word'])
aaa.to_csv(r'no_hypo_wn.csv', index = None, header = True)


In [ ]:
print(big_info[3][2])
print(big_info[3][5])

In [ ]:
df_five_info = pd.DataFrame(big_info, columns = ['Name_MeSH', 'def_MeSH', 'word_list_def_MeSH', 'Name_WN', 'def_WN', 'word_list_def_WN', 'Similarity_highest'])
df_five_info.head()
df_five_info.to_csv(r'sim_sif_with_concept_a_verifie.csv', index = None, header = True)

In [ ]:
import operator as op

def find_name_concept_by_sentence(all_info):
    five_info = []
    for i in range(len(sentences1)):
        if op.eq(dataConceptMeSH.iloc[i].scopeNote, all_info[i][0]):
            five_info.append(dataConceptMeSH.iloc[i].name)
            five_info.append(dataConceptMeSH.iloc[i].scopeNote)
        else:
            continue
    for i in range(len(sentences2)):
        if op.eq(dataConceptWN.iloc[i].definition, all_info[i][1]):
            five_info.append(dataConceptWN.iloc[i].name)
            five_info.append(dataConceptWN.iloc[i].definition)
        else:
            continue
    return five_info
    

In [ ]:
five = find_name_concept_by_sentence(df_five_info)

In [ ]:
'wassup' in model.wv.vocab

In [ ]:
dataaa = pd.read_csv('../Data_extracted/data_using_sim_sif_with_concept_a_verifie.csv', error_bad_lines= False, encoding='utf-8', names = ["Name_MeSH", "def_MeSH", "word_list_def_MeSH", "Name_WN", "def_WN", "word_list_def_WN", "Similarity_highest"])


In [ ]:
dataaa.head()